In [8]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [12]:

PATHS = {
    'M0': '../output/results/M0_results.csv',
    'M1': '../output/results/M1_results.csv',
    'M2': '../output/results/M2_results.csv',
}

# Adjust these to match your data
DATE_COL = 'date'
COUNTRY_COL = 'country'
DD_COL = 'distance_to_distress'
CDS_COL = 'cds_spread'

# Horizons in weeks for computing changes
HORIZONS = [1, 2, 4, 8]

# Define your groups — edit these lists to match your sample
EXPORTERS = [
    'Saudi Arabia',
    'Abu Dhabi',
    'Dubai',
    'Qatar',
    'Colombia',
    'Mexico',
    'Brazil',
    'Egypt',
    'Malaysia']

CONTROLS = None

In [13]:
# ──────────────────────────────────────────────────────────────
# 2. LOAD DATA
# ──────────────────────────────────────────────────────────────

def load_models(paths):
    dfs = {}
    for m, p in paths.items():
        df = pd.read_csv(p, parse_dates=[DATE_COL])
        df = df.sort_values([COUNTRY_COL, DATE_COL])
        dfs[m] = df
    return dfs

dfs = load_models(PATHS)

# Auto-detect controls if not specified
all_countries = sorted(dfs['M0'][COUNTRY_COL].unique())
if CONTROLS is None:
    CONTROLS = [c for c in all_countries if c not in EXPORTERS]

print(f"Total countries: {len(all_countries)}")
print(f"Exporters ({len(EXPORTERS)}): {EXPORTERS}")
print(f"Controls  ({len(CONTROLS)}): {CONTROLS}")

Total countries: 17
Exporters (9): ['Saudi Arabia', 'Abu Dhabi', 'Dubai', 'Qatar', 'Colombia', 'Mexico', 'Brazil', 'Egypt', 'Malaysia']
Controls  (8): ['Chile', 'China', 'Indonesia', 'Philippines', 'South Africa', 'South Korea', 'Thailand', 'Turkey']


In [14]:
def compute_changes(df, horizon_weeks):
    """Compute ΔDD and ΔCDS over a given horizon (in weeks)."""
    out = []
    for country, grp in df.groupby(COUNTRY_COL):
        g = grp.set_index(DATE_COL).sort_index()
        # Resample to weekly if daily data
        g = g[[DD_COL, CDS_COL]].resample('W').last().dropna()
        g[f'delta_dd'] = g[DD_COL].diff(horizon_weeks)
        g[f'delta_cds'] = g[CDS_COL].diff(horizon_weeks)
        g[COUNTRY_COL] = country
        out.append(g.dropna(subset=['delta_dd', 'delta_cds']).reset_index())
    if not out:
        return pd.DataFrame()
    return pd.concat(out, ignore_index=True)

# ──────────────────────────────────────────────────────────────
# 4. CORRELATION FUNCTIONS
# ──────────────────────────────────────────────────────────────

def pearson_corr(dd, cds):
    if len(dd) < 10:
        return np.nan
    return np.corrcoef(dd, cds)[0, 1]


def spearman_corr(dd, cds):
    if len(dd) < 10:
        return np.nan
    rho, _ = spearmanr(dd, cds)
    return rho


def country_correlations(df, corr_func):
    """Compute correlation per country, return Series."""
    results = {}
    for country, grp in df.groupby(COUNTRY_COL):
        dd = grp['delta_dd'].values
        cds = grp['delta_cds'].values
        mask = np.isfinite(dd) & np.isfinite(cds)
        if mask.sum() >= 10:
            results[country] = corr_func(dd[mask], cds[mask])
    return pd.Series(results)


def run_correlation_table(corr_func, corr_name):
    """
    Returns a DataFrame:
        rows = horizons
        columns = MultiIndex (model, group)
        values = mean correlation across countries in that group
    """
    records = []
    for h in HORIZONS:
        for m in PATHS.keys():
            changes = compute_changes(dfs[m], h)
            if changes.empty:
                continue
            corrs = country_correlations(changes, corr_func)

            all_mean = corrs.mean()
            exp_mean = corrs[corrs.index.isin(EXPORTERS)].mean()
            ctrl_mean = corrs[corrs.index.isin(CONTROLS)].mean()

            records.append({
                'horizon_w': h,
                'model': m,
                'all': all_mean,
                'exporters': exp_mean,
                'controls': ctrl_mean,
                'diff': exp_mean - ctrl_mean,
            })

    df_out = pd.DataFrame(records)
    df_out['corr_type'] = corr_name
    return df_out

In [15]:
pearson_results = run_correlation_table(pearson_corr, 'Pearson')
# Spearman
spearman_results = run_correlation_table(spearman_corr, 'Spearman')

results = pd.concat([pearson_results, spearman_results], ignore_index=True)


In [16]:
# 6. DISPLAY TABLES
# ──────────────────────────────────────────────────────────────

def display_table(df, corr_name, group_col, title):
    subset = df[df['corr_type'] == corr_name]
    pivot = subset.pivot_table(index='horizon_w', columns='model', values=group_col)
    pivot = pivot[list(PATHS.keys())]  # enforce column order
    pivot.index.name = 'Horizon (weeks)'
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"  {corr_name} correlation — average across countries")
    print(f"{'='*60}")
    print(pivot.round(3).to_string())
    return pivot


# Table 1: Pearson, all countries
t1 = display_table(results, 'Pearson', 'all',
                    'Pearson — All Countries')

# Table 2: Pearson, exporters vs controls
t2_exp = display_table(results, 'Pearson', 'exporters',
                        'Pearson — Exporters')
t2_ctrl = display_table(results, 'Pearson', 'controls',
                         'Pearson — Controls')

# Table 3: Spearman, exporters vs controls
t3_exp = display_table(results, 'Spearman', 'exporters',
                        'Spearman — Exporters')
t3_ctrl = display_table(results, 'Spearman', 'controls',
                         'Spearman — Controls')


  Pearson — All Countries
  Pearson correlation — average across countries
model               M0     M1     M2
Horizon (weeks)                     
1               -0.064 -0.188 -0.135
2               -0.088 -0.229 -0.199
4               -0.112 -0.260 -0.247
8               -0.131 -0.319 -0.298

  Pearson — Exporters
  Pearson correlation — average across countries
model               M0     M1     M2
Horizon (weeks)                     
1               -0.071 -0.203 -0.145
2               -0.098 -0.255 -0.213
4               -0.132 -0.301 -0.273
8               -0.152 -0.362 -0.328

  Pearson — Controls
  Pearson correlation — average across countries
model               M0     M1     M2
Horizon (weeks)                     
1               -0.055 -0.172 -0.123
2               -0.077 -0.200 -0.183
4               -0.090 -0.214 -0.217
8               -0.107 -0.270 -0.264

  Spearman — Exporters
  Spearman correlation — average across countries
model               M0     M1     M2
Hori

## Correlation with other variables?

In [26]:
# ──────────────────────────────────────────────────────────────
# HORSE RACE: ΔDD vs raw macro variables as predictors of ΔCDS
# ──────────────────────────────────────────────────────────────

# 1. Load macro variables
cca_raw = pd.read_csv('../data/processed/CCA/cca_newfx_rates.csv', parse_dates=['date'])
vix = pd.read_csv('../data/processed/Macroeconomic_variables/VIXCLS.csv', parse_dates=['Date'])
vix.rename(columns={'Date': 'date', 'VIXCLS': 'VIX'}, inplace=True)

# Oil price — reuse from CCA panel or load separately
oil = pd.read_csv('../data/processed/Oil/oil_prices_datastream.csv', parse_dates=['date'])
oil_col = 'price' if 'price' in oil.columns else oil.columns[1]
oil = oil[['date', oil_col]].rename(columns={oil_col: 'oil_price'})

macro = cca_raw[['date', 'country', 'fx_rate', 'cds_spread_5Y']].copy()
macro.rename(columns={'cds_spread_5Y': 'cds_spread'}, inplace=True)
macro = macro.merge(oil, on='date', how='left')
macro = macro.merge(vix[['date', 'VIX']], on='date', how='left')

# Resample to weekly per country
macro.set_index('date', inplace=True)
macro = (
    macro.groupby('country')
    .resample('W').last()
    .reset_index()
)
macro[['oil_price', 'VIX']] = macro.groupby('country')[['oil_price', 'VIX']].ffill()

# 3. Merge DD from each model
for m, path in PATHS.items():
    mdf = pd.read_csv(path, parse_dates=[DATE_COL])
    mdf = mdf[[DATE_COL, COUNTRY_COL, DD_COL]].rename(columns={DD_COL: f'dd_{m}'})
    mdf.set_index([DATE_COL, COUNTRY_COL], inplace=True)
    mdf = mdf.groupby(COUNTRY_COL).resample('W', level=DATE_COL).last().reset_index()
    macro = macro.merge(mdf, on=['date', 'country'], how='left')


# 4. Compute correlations per country and horizon
MACRO_VARS = {
    'FX':  'fx_rate',
    'Oil': 'oil_price',
    'VIX': 'VIX',
}

def horse_race(horizon, corr_func=pearson_corr):
    rows = []
    for country, g in macro.groupby('country'):
        g = g.sort_values('date').copy()

        delta_cds = g['cds_spread'].diff(horizon).values

        row = {'country': country}

        # DD from each model
        for m in PATHS.keys():
            delta_dd = g[f'dd_{m}'].diff(horizon).values
            mask = np.isfinite(delta_dd) & np.isfinite(delta_cds)
            row[f'DD({m})'] = corr_func(delta_dd[mask], delta_cds[mask]) if mask.sum() >= 10 else np.nan

        # Raw macro variables
        for name, col in MACRO_VARS.items():
            delta_macro = g[col].diff(horizon).values
            mask = np.isfinite(delta_macro) & np.isfinite(delta_cds)
            row[name] = corr_func(delta_macro[mask], delta_cds[mask]) if mask.sum() >= 10 else np.nan

        rows.append(row)

    return pd.DataFrame(rows).set_index('country')

# 5. Run and display
for h in HORIZONS:
    hr = horse_race(h)

    exp_avg = hr[hr.index.isin(EXPORTERS)].mean()
    ctrl_avg = hr[hr.index.isin(CONTROLS)].mean()
    all_avg = hr.mean()

    summary = pd.DataFrame({
        'All': all_avg,
        'Exporters': exp_avg,
        'Controls': ctrl_avg,
        'Exp−Ctrl': exp_avg - ctrl_avg,
    }).round(3)

    print(f"\n{'='*65}")
    print(f"  Horse Race — Pearson corr(Δx, ΔCDS), horizon = {h}w")
    print(f"{'='*65}")
    print(summary.to_string())


  Horse Race — Pearson corr(Δx, ΔCDS), horizon = 1w
          All  Exporters  Controls  Exp−Ctrl
DD(M0) -0.064     -0.071    -0.055    -0.016
DD(M1) -0.188     -0.203    -0.172    -0.031
DD(M2) -0.135     -0.145    -0.123    -0.021
FX      0.266      0.275     0.401    -0.126
Oil    -0.136     -0.204    -0.177    -0.028
VIX     0.265      0.324     0.348    -0.024

  Horse Race — Pearson corr(Δx, ΔCDS), horizon = 2w
          All  Exporters  Controls  Exp−Ctrl
DD(M0) -0.088     -0.098    -0.077    -0.021
DD(M1) -0.229     -0.255    -0.200    -0.054
DD(M2) -0.199     -0.213    -0.183    -0.030
FX      0.280      0.290     0.415    -0.125
Oil    -0.176     -0.279    -0.209    -0.070
VIX     0.332      0.407     0.444    -0.037

  Horse Race — Pearson corr(Δx, ΔCDS), horizon = 4w
          All  Exporters  Controls  Exp−Ctrl
DD(M0) -0.112     -0.132    -0.090    -0.041
DD(M1) -0.260     -0.301    -0.214    -0.087
DD(M2) -0.247     -0.273    -0.217    -0.056
FX      0.307      0.303     0.

In [ ]:

# 2. Build macro panel: one row per (date, country) with FX, and global series (oil, VIX)
macro = cca_raw[['date', 'country', 'fx_rate', 'cds_spread_5Y']].copy()
macro.rename(columns={'cds_spread_5Y': 'cds_spread'}, inplace=True)
macro = macro.merge(oil, on='date', how='left')
macro = macro.merge(vix[['date', 'VIX']], on='date', how='left')

# Resample to weekly per country
macro.set_index('date', inplace=True)
macro = (
    macro.groupby('country')
    .resample('W').last()
    .drop(columns='country')
    .reset_index()
)
macro[['oil_price', 'VIX']] = macro.groupby('country')[['oil_price', 'VIX']].ffill()

# 3. Merge DD from each model
for m, path in PATHS.items():
    mdf = pd.read_csv(path, parse_dates=[DATE_COL])
    mdf = mdf[[DATE_COL, COUNTRY_COL, DD_COL]].rename(columns={DD_COL: f'dd_{m}'})
    mdf.set_index([DATE_COL, COUNTRY_COL], inplace=True)
    mdf = mdf.groupby(COUNTRY_COL).resample('W', level=DATE_COL).last().reset_index()
    macro = macro.merge(mdf, on=['date', 'country'], how='left')

# 4. Compute correlations per country and horizon
MACRO_VARS = {
    'FX':  'fx_rate',
    'Oil': 'oil_price',
    'VIX': 'VIX',
}

def horse_race(horizon, corr_func=pearson_corr):
    rows = []
    for country, g in macro.groupby('country'):
        g = g.sort_values('date').copy()

        delta_cds = g['cds_spread'].diff(horizon).values

        row = {'country': country}

        # DD from each model
        for m in PATHS.keys():
            delta_dd = g[f'dd_{m}'].diff(horizon).values
            mask = np.isfinite(delta_dd) & np.isfinite(delta_cds)
            row[f'DD({m})'] = corr_func(delta_dd[mask], delta_cds[mask]) if mask.sum() >= 10 else np.nan

        # Raw macro variables
        for name, col in MACRO_VARS.items():
            delta_macro = g[col].diff(horizon).values
            mask = np.isfinite(delta_macro) & np.isfinite(delta_cds)
            row[name] = corr_func(delta_macro[mask], delta_cds[mask]) if mask.sum() >= 10 else np.nan

        rows.append(row)

    return pd.DataFrame(rows).set_index('country')

# 5. Run and display
for h in HORIZONS:
    hr = horse_race(h)

    exp_avg = hr[hr.index.isin(EXPORTERS)].mean()
    ctrl_avg = hr[hr.index.isin(CONTROLS)].mean()
    all_avg = hr.mean()

    summary = pd.DataFrame({
        'All': all_avg,
        'Exporters': exp_avg,
        'Controls': ctrl_avg,
        'Exp−Ctrl': exp_avg - ctrl_avg,
    }).round(3)

    print(f"\n{'='*65}")
    print(f"  Horse Race — Pearson corr(Δx, ΔCDS), horizon = {h}w")
    print(f"{'='*65}")
    print(summary.to_string())